---
# `Positional Encoding`
---

Introduction
- It is Step 3 of transformer architecture
- It is done so that we can maintian the Order in the Sentence that how the word order is their in the sentence.
- So, Positional encoding adds more information to previous semantic meaning as the word order 

eg. 
1. The Dog chased Cat
2. The Cat chased Dog

For such sentences, if we are not applying positional encoding. Then we will consider Both sentence same.
Some values to get exact embedding with the position info as well


---
---
# Detailed Notes
---
---

# Positional Encoding in Transformers

## What is Positional Encoding?

**Positional Encoding** is a technique used in Transformers to provide information about the **position of each token** in a sequence.

Unlike RNNs and LSTMs, which process words one after another, Transformers process **all tokens simultaneously (in parallel)**. Because of this, they have **no inherent understanding of word order**.

Positional encoding solves this problem by injecting position information into the token embeddings.

---

# Why Do We Need Positional Encoding?

Consider these two sentences:

```text
Dog bites man
```

and

```text
Man bites dog
```

Both sentences contain the same words.

Without positional information, the Transformer would see the same set of token embeddings and would struggle to distinguish between them.

Word order completely changes the meaning.

---

# Problem Without Positional Encoding

Suppose we have:

```text
I love AI
```

After tokenization:

```text
["I", "love", "AI"]
```

After embedding:

```text
I

↓

[0.2,0.6,0.1]

love

↓

[0.8,0.4,-0.3]

AI

↓

[-0.2,0.9,0.7]
```

The Transformer receives:

```text
[0.2,0.6,0.1]

[0.8,0.4,-0.3]

[-0.2,0.9,0.7]
```

These vectors contain **semantic meaning**, but they do **not** indicate which word is first, second, or third.

---

# Transformer Pipeline

```text
Input Text
      │
      ▼
Tokenization
      │
      ▼
Token IDs
      │
      ▼
Token Embeddings
      │
      ▼
Positional Encoding
      │
      ▼
Final Input Embeddings
      │
      ▼
Transformer Layers
```

---

# Basic Idea

Each position receives a unique vector.

Example:

```text
Position 0

↓

[0.1,0.3,0.7]

Position 1

↓

[0.5,-0.2,0.4]

Position 2

↓

[-0.1,0.8,0.2]
```

These vectors are added to the token embeddings.

---

# Example

Sentence:

```text
I love AI
```

Token Embeddings

```text
I

↓

[0.2,0.4,0.1]

love

↓

[0.7,0.5,-0.2]

AI

↓

[-0.3,0.8,0.6]
```

Position Embeddings

```text
Position 0

↓

[0.1,0.1,0.1]

Position 1

↓

[0.2,0.2,0.2]

Position 2

↓

[0.3,0.3,0.3]
```

Final Input

```text
I

↓

[0.3,0.5,0.2]

love

↓

[0.9,0.7,0.0]

AI

↓

[0.0,1.1,0.9]
```

The Transformer now knows:

* What the token means (embedding)
* Where it appears (position)

---

# Formula

The original Transformer paper introduced **sinusoidal positional encoding**.

For even dimensions:

[
$PE(pos,2i)=\sin\left(\frac{pos}{10000^{2i/d}}\right)$
]

For odd dimensions:

[
$PE(pos,2i+1)=\cos\left(\frac{pos}{10000^{2i/d}}\right)$
]

where:

* **pos** = token position
* **i** = embedding dimension index
* **d** = embedding size

---

# Why Sine and Cosine?

The sinusoidal functions provide several useful properties:

* Every position gets a unique representation.
* Nearby positions have similar encodings.
* Relative distances between positions can be inferred.
* The model can generalize to sequence lengths longer than those seen during training (within practical limits).

---

# Example of Sinusoidal Encoding

Suppose the embedding dimension is **4**.

| Position | Dimension 0 | Dimension 1 | Dimension 2 | Dimension 3 |
| -------- | ----------: | ----------: | ----------: | ----------: |
| 0        |       0.000 |       1.000 |       0.000 |       1.000 |
| 1        |       0.841 |       0.540 |       0.010 |       0.999 |
| 2        |       0.909 |      -0.416 |       0.020 |       0.999 |
| 3        |       0.141 |      -0.990 |       0.030 |       0.999 |

Each row is added to the corresponding token embedding.

---

# Visual Representation

```text
Sentence

I      Love      AI

│        │        │

▼        ▼        ▼

Embedding Embedding Embedding

│        │        │

+

+

+

│        │        │

▼        ▼        ▼

Position 0 Position 1 Position 2

│        │        │

▼        ▼        ▼

Final Input Embeddings

│

▼

Transformer
```

---

# Learned Positional Embeddings

Many modern Transformers (such as GPT variants and BERT) use **learned positional embeddings** instead of fixed sinusoidal encodings.

Instead of computing positions with sine and cosine, the model learns a position embedding matrix during training.

Example:

| Position | Learned Embedding        |
| -------: | ------------------------ |
|        0 | [0.12, -0.44, 0.81, ...] |
|        1 | [0.31, 0.52, -0.19, ...] |
|        2 | [-0.08, 0.66, 0.40, ...] |

These vectors are optimized through backpropagation along with the rest of the model.

---

# Implementation (TensorFlow/Keras)

## Token Embedding

```python
from tensorflow.keras.layers import Embedding

vocab_size = 10000
embedding_dim = 128

token_embedding = Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim
)
```

---

## Learned Position Embedding

```python
from tensorflow.keras.layers import Embedding
import tensorflow as tf

max_length = 100
embedding_dim = 128

position_embedding = Embedding(
    input_dim=max_length,
    output_dim=embedding_dim
)

positions = tf.range(start=0, limit=max_length, delta=1)

position_vectors = position_embedding(positions)

print(position_vectors.shape)
```

Output:

```text
(100, 128)
```

---

## Add Token and Position Embeddings

```python
import tensorflow as tf

token_vectors = token_embedding(
    tf.constant([[15, 89, 700]])
)

position_vectors = position_embedding(tf.range(3))

final_embeddings = token_vectors + position_vectors

print(final_embeddings.shape)
```

Output:

```text
(1, 3, 128)
```

---

# Implementation (PyTorch)

```python
import torch
import torch.nn as nn

vocab_size = 10000
embedding_dim = 128
max_len = 100

token_embedding = nn.Embedding(vocab_size, embedding_dim)
position_embedding = nn.Embedding(max_len, embedding_dim)

tokens = torch.tensor([[15, 89, 700]])

positions = torch.arange(3)

token_vectors = token_embedding(tokens)
position_vectors = position_embedding(positions)

final_embeddings = token_vectors + position_vectors

print(final_embeddings.shape)
```

Output:

```text
torch.Size([1, 3, 128])
```

---

# Sinusoidal vs Learned Positional Embeddings

| Feature                            | Sinusoidal  | Learned                                           |
| ---------------------------------- | ----------- | ------------------------------------------------- |
| Trainable                          | ❌ No        | ✅ Yes                                             |
| Parameters                         | None        | Additional parameters                             |
| Generalization to Longer Sequences | Better      | Limited to trained maximum length unless extended |
| Used in Original Transformer       | ✅ Yes       | ❌ No                                              |
| Used in Modern LLMs                | Less Common | More Common                                       |

---

# Advantages

* Preserves word order.
* Enables parallel processing.
* Allows attention to distinguish between different positions.
* Improves understanding of sentence structure.

---

# Complete Transformer Input

For a decoder-only model (e.g., GPT):

[
$\text{Input}$
=
$\text{Token Embedding}$
+
$\text{Position Embedding}$
]

For BERT:

[
$\text{Input}$
=
$\text{Token Embedding}$
+
$\text{Position Embedding}$
+
$\text{Segment Embedding}$
]

---

# Complete Pipeline

```text
Input Text

↓

Tokenizer

↓

Token IDs

↓

Token Embeddings

↓

Positional Embeddings

↓

Addition

↓

Final Input Embeddings

↓

Transformer Blocks

↓

Prediction
```

---

# Interview Summary

> **Positional Encoding** provides Transformers with information about the order of tokens in a sequence. Since Transformers process all tokens in parallel, they do not naturally know which token comes first or last. Positional information is added to token embeddings before they enter the Transformer. The original Transformer used fixed sinusoidal positional encodings, while many modern models use learned positional embeddings. This allows the model to capture both the meaning of each token and its position in the sequence, enabling it to understand sentence structure effectively.
